In [4]:
from openrouter import OpenRouter
from dotenv import load_dotenv  
import os

load_dotenv()
api_key=os.getenv("OPENROUTER_API_KEY")

with OpenRouter(
  api_key=os.getenv("OPENROUTER_API_KEY", ""),
) as client:
  response = client.chat.send(
    model="qwen/qwen3-32b",
    messages=[
      {
        "role": "user",
        "content": "Hi"
      }
    ]
  )

  print(response.choices[0].message.content)

Hello! How can I assist you today? 😊


In [ ]:
SYSTEM_PROMPT = """

Role:
คุณเป็นผู้เชี่ยวชาญด้านการวิเคราะห์ SMS หลอกลวง ในประเทศไทย และเป็นหน่วยรักษาความปลอดภัยทางไซเบอร์มานานกว่า 50 ปี

Task:
โมเดล NLP ตรวจพบว่าข้อความนี้มีแนวโน้มเป็นข้อความหลอกลวง
หน้าที่ของคุณคือวิเคราะห์และอธิบายเหตุผลว่าทำไมข้อความนี้จึงเป็น scam

Analysis method:
1. ระบุจุดน่าสงสัยในข้อความ เช่น การส่งลิงก์ปลอม, การแอบอ้าง, การให้ข้อมูลเท็จ
2. อ้างอิงนโยบายจริงขององค์กรหรือธนาคารที่เกี่ยวข้องจาก Context ที่ให้มา
3. แสดงให้เห็นถึงความขัดแย้งระหว่างข้อความกับนโยบายจริง

Response format:
ตอบเป็นภาษาไทย กระชับ เข้าใจง่าย และใช้โครงสร้างตามดังนี้:
- จุดที่น่าสงสัย (ระบุจุดน่าสงสัยที่พบ)
- นโยบายที่เกี่ยวข้อง (อ้างอิงจาก Context)
- สรุป (อธิบายสั้นๆ ว่าทำไมเป็น Scam)

Important:
- ให้ใช้เฉพาะข้อมูลจาก Context ที่ให้มาเท่านั้น ห้ามแต่งนโยบายขึ้นมาเอง
- ถ้าไม่มีนโยบายที่เกี่ยวข้องใน Context ให้วิเคราะห์จากลักษณะของข้อความแทน
"""

In [ ]:
from openrouter import OpenRouter
from dotenv import load_dotenv
import os

load_dotenv()

def analyze_scam_sms(sms_message: str, retriever, api_key: str) -> str:

    retrieve_docs = retriever.invoke(sms_message)
    context = "\n\n---\n\n".join([
        f"[ที่มา: {doc.metadata['source']}]\n{doc.page_content}"
        for doc in retrieve_docs
    ])

    # 2. สร้าง prompt
    user_prompt = USER_PROMPT_TEMPLATE.format(
        sms_message=sms_message,
        retrieved_context=context
    )

    # 3. ส่งให้ LLM ผ่าน OpenRouter
    with OpenRouter(api_key=api_key) as client:
        response = client.chat.send(
            model="qwen/qwen3-32b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ]
        )

    return response.choices[0].message.content